In [ ]:
import numpy as np
from scipy.io import loadmat
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelBinarizer
from sklearn.metrics import roc_curve, auc

In [ ]:
data=loadmat("data/PaviaU.mat")['paviaU']
labels=loadmat("data/PaviaU_gt.mat")['paviaU_gt']

In [ ]:
class PrincipalComponentAnalysis:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        self.mean = np.mean(X, axis=0)
        X = X - self.mean
        cov = np.cov(X.T)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        eigenvectors = eigenvectors.T
        idxs = np.argsort(-eigenvalues)
        eigenvalues = eigenvalues[idxs]
        eigenvectors = eigenvectors[idxs]
        self.components = eigenvectors[:self.n_components]
        total_variance = np.sum(eigenvalues)
        explained_variance = eigenvalues[:self.n_components]
        self.explained_variance_ratio_ = explained_variance / total_variance

    def transform(self, X):
        X = X - self.mean
        return np.dot(X, self.components.T)
    # New function
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

     # New function
    def plot_explained_variance_ratio(self):
        if self.explained_variance_ratio_ is not None:
            plt.bar(range(1, self.n_components + 1), self.explained_variance_ratio_)
            plt.xlabel('Principal Component')
            plt.ylabel('Explained Variance Ratio')
            plt.title('Explained Variance Ratio for Principal Components')
            plt.show()
        else:
            print("Please fit the PCA model first to compute explained variance ratio.")


In [ ]:
class NaiveBayes:

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._classes = np.unique(y)
        n_classes = len(self._classes)

        # calculate mean, var, and prior for each class
        self._mean = np.zeros((n_classes, n_features), dtype=np.float64)
        self._var = np.zeros((n_classes, n_features), dtype=np.float64)
        self._priors = np.zeros(n_classes, dtype=np.float64)

        for idx, c in enumerate(self._classes):
            X_c = X[y == c]
            self._mean[idx, :] = X_c.mean(axis=0)
            self._var[idx, :] = X_c.var(axis=0)
            self._priors[idx] = X_c.shape[0] / float(n_samples)
        #self._priors = self.generate_random_priors(n_classes)

    def predict(self, X):
        y_pred = [self._predict(x) for x in X]
        return np.array(y_pred)

    def _predict(self, x):
        posteriors = []

        # calculate posterior probability for each class
        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            posterior = np.sum(np.log(self._pdf(idx, x)))
            posterior = posterior + prior
            posteriors.append(posterior)

        # return class with the highest posterior
        return self._classes[np.argmax(posteriors)]

    def _pdf(self, class_idx, x):
        mean = self._mean[class_idx]
        var = self._var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator
    def accuracy(self, y, X):
        print(len(y),len(X))
        accuracy = np.sum(y == X) / len(y)
        return accuracy
    def ClassRemove(self, c, X, y):
        class_to_remove = c
        X_filtered = X[y != class_to_remove]
        y_filtered = y[y != class_to_remove]
        return X_filtered, y_filtered
    def predict_proba(self, X):
        n_samples = X.shape[0]
        class_probs = np.zeros((n_samples, len(self._classes)))

        for idx, c in enumerate(self._classes):
            prior = np.log(self._priors[idx])
            class_probs[:, idx] = prior + np.sum(np.log(self._pdf(idx, X)), axis=1)

        class_probs = np.exp(class_probs - class_probs.max(axis=1)[:, np.newaxis])
        class_probs /= class_probs.sum(axis=1)[:, np.newaxis]

        return class_probs
    def generate_random_priors(self,num_classes):
      n_classes = len(self._classes)
      priors = np.random.rand(num_classes)
      priors /= priors.sum()
      return priors

In [ ]:
class Evaluating_Models:
    def __init__(self, model):
      self._precision = []
      self._recall = []
      self._accuracy = None
      self._TP = []
      self._TN = []
      self._FP = []
      self._FN = []
      self._TPR = []
      self._FPR = []
      self.F1_score = []
      self._Model = model
      self.conf_matrix = None
      self.best_components = None
    def calculate_confusion_matrix(self, y_test, y_pred):
      self.conf_matrix = confusion_matrix(y_test, y_pred)
      for idx in range(len(self.conf_matrix[0])):
        self._TP.append(self.conf_matrix[idx,idx])
        self._TN.append(np.sum(np.diag(self.conf_matrix)) - self.conf_matrix[idx,idx])
        self._FP.append(np.sum(self.conf_matrix[ : ,idx]) - self.conf_matrix[idx,idx])
        self._FN.append(np.sum(self.conf_matrix[idx, : ]) - self.conf_matrix[idx,idx])
        self._TPR.append(self._TP[idx] / (self._TP[idx] + self._FN[idx]))
        self._FPR.append(self._FP[idx] / (self._FP[idx] + self._TN[idx]))
        self._precision.append(self._TP[idx] / (self._TP[idx] + self._FP[idx] + 1e-9))
        self._recall.append(self._TP[idx] / (self._TP[idx] + self._FN[idx] + 1e-9))
        self.F1_score.append(2 * (self._precision[idx] * self._recall[idx]) / (self._precision[idx] + self._recall[idx] + 1e-9))
      self._accuracy = np.sum(y_pred == y_test) / len(y_test)

    def plot_confusion_matrix(self, classes):
      _class = classes
      plt.figure(figsize=(8, 6))
      sns.heatmap(self.conf_matrix, annot=True, fmt="d", cbar=False, xticklabels = _class, yticklabels = _class)
      plt.xlabel("Predicted")
      plt.ylabel("Actual")
      plt.title("Confusion Matrix")
      plt.show()

    def cross_val(self, X, y, num_folds=5, random_state=123):
            accuracies = []
            precisions = []
            recalls = []
            f1_scores = []

            skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=random_state)
            pca = PrincipalComponentAnalysis(self.best_components)
            tmp_X = pca.fit_transform(X)
            for train_index, test_index in skf.split(tmp_X , y):
                X_train, X_test = X[train_index], X[test_index]
                y_train, y_test = y[train_index], y[test_index]

                clf = self._Model()
                clf.fit(X_train, y_train)
                y_pred = clf.predict(X_test)

                self.calculate_confusion_matrix(y_test, y_pred)

                accuracies.append(self._accuracy)
                precisions.append(self._precision)
                recalls.append(self._recall)
                f1_scores.append(self.F1_score)

            avg_accuracy = np.mean(accuracies)
            avg_precision = np.mean(precisions)
            avg_recall = np.mean(recalls)
            avg_f1_score = np.mean(f1_scores)

            std_accuracy = np.std(accuracies)
            std_precision = np.std(precisions)
            std_recall = np.std(recalls)
            std_f1_score = np.std(f1_scores)

            metrics = {
                    'accuracy': (avg_accuracy, std_accuracy),
                    'precision': (avg_precision, std_precision),
                    'recall': (avg_recall, std_recall),
                    'f1_score': (avg_f1_score, std_f1_score),
                }
            return metrics
    def ROC_AUC(self, X_train, y_train, X_test, y_test):
      clf = self._Model()
      clf.fit(X_train, y_train)
      classes = ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets', 'Bare Soil', 'Bitumen', 'Self-Blocking Bricks', 'Shadows']

      lb = LabelBinarizer()
      y_test_binary = lb.fit_transform(y_test)

      # Compute ROC curve and ROC AUC for each class separately
      fpr = dict()
      tpr = dict()
      roc_auc = dict()
      n_classes = len(classes)

      y_score = clf.predict_proba(X_test)  # Use predict_proba to get class probabilities

      for i in range(n_classes):
          fpr[i], tpr[i], _ = roc_curve(y_test_binary[:, i], y_score[:, i])
          roc_auc[i] = auc(fpr[i], tpr[i])

      # Plot ROC curves and AUC for each class
      plt.figure(figsize=(10, 6))
      for i in range(n_classes):
          plt.plot(fpr[i], tpr[i], label=f'Class {classes[i]} (AUC = {roc_auc[i]:.2f})')

      plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
      plt.xlim([0.0, 1.0])
      plt.ylim([0.0, 1.05])
      plt.xlabel('False Positive Rate')
      plt.ylabel('True Positive Rate')
      plt.title('Multi-Class ROC Curve')
      plt.legend(loc="lower right")
      plt.show()

    def init(self):
        self._precision = []
        self._recall = []
        self._TP = []
        self._TN = []
        self._FP = []
        self._FN = []
        self._TPR = []
        self._FPR = []
        self.F1_score = []
        self.cumulative_explained_variance_ratio = []
    def Suitable_Components(self, X, y):
        tmp_model = self._Model()
        X, y = tmp_model.ClassRemove(0, X, y)
        pca = PrincipalComponentAnalysis(self.best_components)
        tmp_X = pca.fit_transform(X)
        stratified_split = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=123)
        for train_index, test_index in stratified_split.split(tmp_X, y):
            X_train, X_test = tmp_X[train_index], tmp_X[test_index]
            y_train, y_test = y[train_index], y[test_index]
        tmp_model.fit(X_train, y_train)
        y_pred = tmp_model.predict(X_test)
        self.calculate_confusion_matrix(y_test, y_pred)
        return X_train, y_train, X_test, y_test
    def Find_Suitable_Components(self, a, b, X, y):
      tmp_model = self._Model()
      X, y = tmp_model.ClassRemove(0, X, y)
      accuracy = []
      cumulative_explained_variance_ratio = []
      for i in range(a, b + 1):
        self.init()
        pca = PrincipalComponentAnalysis(i)
        print(i)
        tmp_X = pca.fit_transform(X)
        self.cumulative_explained_variance_ratio.append(np.sum(pca.explained_variance_ratio_))
        k = 5
        stratified_split = StratifiedShuffleSplit(n_splits=k, test_size=0.2, random_state=123)
        for train_index, test_index in stratified_split.split(tmp_X, y):
          X_train, X_test = tmp_X[train_index], tmp_X[test_index]
          y_train, y_test = y[train_index], y[test_index]
        tmp_model.fit(X_train, y_train)
        y_pred = tmp_model.predict(X_test)
        self.calculate_confusion_matrix(y_test, y_pred)
        accuracy.append(self._accuracy)
        cumulative_explained_variance_ratio.append(self.cumulative_explained_variance_ratio)
        print(self._accuracy)
        classes = ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets',
            'Bare Soil', 'Bitumen', 'Self-Blocking Bricks', 'Shadows']
      best_accuracy_index = np.argmax(accuracy)
      self.best_components = best_accuracy_index + 1
      print("best_components:",self.best_components)
      fig, axs = plt.subplots(1, 2, figsize=(12, 5))
      axs[0].plot(range(a, b + 1), accuracy, marker='o')
      axs[0].set_title('Model Accuracy vs. Number of Components')
      axs[0].set_xlabel('Number of Components')
      axs[0].set_ylabel('Accuracy')
      axs[0].grid()
      axs[1].plot(range(a, b + 1), cumulative_explained_variance_ratio, color = "Red", marker='o')
      axs[1].set_title('Cumulative Explained Variance Ratio vs. Number of Components')
      axs[1].set_xlabel('Number of Components')
      axs[1].set_ylabel('Cumulative Explained Variance Ratio')
      axs[1].grid()
      plt.tight_layout()
      plt.show()

In [ ]:
X = data.reshape((data.shape[0] * data.shape[1], data.shape[2]))
y = labels.reshape((labels.shape[0] * labels.shape[1]))

In [ ]:
EM = Evaluating_Models(NaiveBayes)
EM.Find_Suitable_Components(1,50, X, y)


In [ ]:
X_train, y_train, X_test, y_test = EM.Suitable_Components(X, y)
conf_matrix = confusion_matrix(y_test, y_pred)

classes = ['Asphalt', 'Meadows', 'Gravel', 'Trees', 'Painted metal sheets',
            'Bare Soil', 'Bitumen', 'Self-Blocking Bricks', 'Shadows']
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cbar=False, xticklabels = classes, yticklabels = classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
EM.ROC_AUC(X_train, y_train, X_test, y_test)

In [ ]:
result = EM.cross_val(X, y, num_folds=5, random_state=123)
precision_mean, precision_std = result['precision']
recall_mean, recall_std = result['recall']
f1_score_mean, f1_score_std = result['f1_score']
metrics = ['Precision', 'Recall', 'F1 Score']
means = [precision_mean, recall_mean, f1_score_mean]
stds = [precision_std, recall_std, f1_score_std]
fig, ax = plt.subplots()
x = range(len(metrics))
ax.bar(x, means, yerr=stds, align='center', alpha=0.5, ecolor='black', capsize=10)
ax.set_ylabel('Score')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_title('Performance Metrics')
ax.yaxis.grid(True)
plt.tight_layout()
plt.show()
